This notebooks serves as a testing playground for the cxllin/Llama2-7b-med-v1 LLM. This model is very small, but has been finetuned on medical data.

Author: Henri Smidt

Email: finn.smidt@stud.uni-heidelberg.de


In [1]:
%pip install --upgrade --quiet  xformers --quiet
%pip install --upgrade --quiet  langchain    --quiet
%pip install --upgrade --quiet  bitsandbytes --quiet
%pip install --upgrade --quiet  python-dotenv --quiet
%pip install accelerate --quiet



  error: subprocess-exited-with-error
  
  × python setup.py bdist_wheel did not run successfully.
  │ exit code: 1
  ╰─> [361 lines of output]
      running bdist_wheel
      running build
      running build_py
      creating build
      creating build/lib.macosx-11.1-arm64-cpython-38
      creating build/lib.macosx-11.1-arm64-cpython-38/xformers
      copying xformers/_deprecation_warning.py -> build/lib.macosx-11.1-arm64-cpython-38/xformers
      copying xformers/attn_bias_utils.py -> build/lib.macosx-11.1-arm64-cpython-38/xformers
      copying xformers/checkpoint.py -> build/lib.macosx-11.1-arm64-cpython-38/xformers
      copying xformers/__init__.py -> build/lib.macosx-11.1-arm64-cpython-38/xformers
      copying xformers/test.py -> build/lib.macosx-11.1-arm64-cpython-38/xformers
      copying xformers/utils.py -> build/lib.macosx-11.1-arm64-cpython-38/xformers
      copying xformers/_cpp_lib.py -> build/lib.macosx-11.1-arm64-cpython-38/xformers
      copying xformers/info.py ->

In [9]:
from langchain_community.llms.huggingface_pipeline import HuggingFacePipeline
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
import os
from dotenv import load_dotenv
import transformers
from torch import cuda, bfloat16

# load environment variables from .env file
from dotenv import load_dotenv, find_dotenv

_ = load_dotenv(find_dotenv())

bitsAndBites_config = transformers.BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=bfloat16,
)

model_id = "meta-llama/Llama-2-13b-chat-hf"
hf_auth = os.environ.get("HF_AUTH")

tokenizer = AutoTokenizer.from_pretrained(model_id, use_auth_token=hf_auth)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    trust_remote_code=True,
    quantization_config=bitsAndBites_config,
    device_map="auto",
    do_sample=True,
    token=hf_auth,
)

model.eval()

/usr/local/lib/python3.10/dist-packages/transformers/models/auto/tokenization_auto.py:671: FutureWarning: The `use_auth_token` argument is deprecated and will be removed in v5 of Transformers. Please use `token` instead.
  warnings.warn(


config.json:   0%|          | 0.00/587 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/33.4k [00:00<?, ?B/s]

model-00001-of-00003.safetensors:   0%|          | 0.00/9.95G [00:00<?, ?B/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/9.90G [00:00<?, ?B/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/6.18G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

/usr/local/lib/python3.10/dist-packages/transformers/utils/hub.py:374: FutureWarning: The `use_auth_token` argument is deprecated and will be removed in v5 of Transformers. Please use `token` instead.
  warnings.warn(


generation_config.json:   0%|          | 0.00/188 [00:00<?, ?B/s]

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(32000, 5120)
    (layers): ModuleList(
      (0-39): 40 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear4bit(in_features=5120, out_features=5120, bias=False)
          (k_proj): Linear4bit(in_features=5120, out_features=5120, bias=False)
          (v_proj): Linear4bit(in_features=5120, out_features=5120, bias=False)
          (o_proj): Linear4bit(in_features=5120, out_features=5120, bias=False)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear4bit(in_features=5120, out_features=13824, bias=False)
          (up_proj): Linear4bit(in_features=5120, out_features=13824, bias=False)
          (down_proj): Linear4bit(in_features=13824, out_features=5120, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm()
        (post_attention_layernorm): LlamaRMSNorm()
      )
    )
    (norm

# Neuer Abschnitt

In [10]:
model.get_memory_footprint()

7083970560

In [11]:
pipe = pipeline(
    task="text-generation",
    model=model,
    return_full_text=True,
    tokenizer=tokenizer,
    max_new_tokens=256,
    temperature=0.01,
    repetition_penalty=1.1,
)

hf = HuggingFacePipeline(pipeline=pipe)

In [15]:
from langchain import PromptTemplate, LLMChain

template = """You are an assistant for medical question-answering tasks. You must use the following two pieces of retrieved context to answer the question. Cite the retrieved context you used to answer the question. If none of the retrieved contexts provide the answer, say that you couldnt find any relevant sources and try to answer the answer without the retrieved documents. However, if you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.
Question: {question}
Context: {context}
Answer:
"""  # Using the context, but not using it if it thinks its wrong.

template = """Generate an answer for the following question. the knowledge you can use can only be derived from the two contexts, that are given. Do not make up the answer yourself.
Question: {question}
Context: {context} """  # Using the context, but not consistently. Sometimes it hallucinates.

template = """Generate an answer for the following question using only these two contexts as your base of knowledge (dont make up your own answer): {context} Say which of the sources you used. Do not state why you used which source.
Question: {question}
"""  # Working!!!

prompt = PromptTemplate(input_variables=["question", "context"], template=template)

chain = LLMChain(prompt=prompt, llm=hf)

question = "What is electroencephalography?"
context = """Source: Wikipedia: Dogs like to run around and play.
Source: Journal of Medicine: Electroencephalography is a method to measure the weight of a patients feet.
"""

print(chain.predict(question=question, context=context))


Answer: According to the Journal of Medicine, electroencephalography is a method to measure the weight of a patient's feet.
